In [1]:
import pandas as pd
import sqlite3

# 1. Ingest the dataset
df = pd.read_csv('HR_Employee_Attrition.csv')

# 2. Transform the categorical attrition data into a binary integer for SQL aggregation
df['Attrition_Flag'] = df['Attrition'].apply(lambda x: 1 if x == 'Yes' else 0)

# 3. Establish the local relational database connection
conn = sqlite3.connect('hr_analytics.db')
df.to_sql('hr_data', conn, if_exists='replace', index=False)

# 4. Execute the Advanced Analytical Query (CTEs & Window Functions)
query_advanced = """
WITH CompanyAverage AS (
    SELECT CAST(SUM(Attrition_Flag) AS FLOAT) / COUNT(Employee_ID) as Avg_Attrition
    FROM hr_data
),
DepartmentStats AS (
    SELECT
        Department,
        COUNT(Employee_ID) as Headcount,
        CAST(SUM(Attrition_Flag) AS FLOAT) / COUNT(Employee_ID) as Dept_Attrition
    FROM hr_data
    GROUP BY Department
)
SELECT
    d.Department,
    d.Headcount,
    ROUND(d.Dept_Attrition * 100, 2) as Dept_Attrition_Pct,
    ROUND((d.Dept_Attrition - c.Avg_Attrition) * 100, 2) as Variance_From_Average,
    RANK() OVER (ORDER BY d.Dept_Attrition DESC) as Risk_Rank
FROM DepartmentStats d, CompanyAverage c;
"""

print("--- EXECUTIVE ATTRITION RISK REPORT ---")
print(pd.read_sql(query_advanced, conn))

--- EXECUTIVE ATTRITION RISK REPORT ---
  Department  Headcount  Dept_Attrition_Pct  Variance_From_Average  Risk_Rank
0      Sales        316               27.22                   5.32          1
1         HR        105               20.95                  -0.95          2
2         IT        145               20.00                  -1.90          3
3    Finance        146               19.18                  -2.72          4
4        R&D        288               18.75                  -3.15          5


In [2]:
# Save the results of our advanced CTE & Window Function query
advanced_results = pd.read_sql(query_advanced, conn)
advanced_results.to_csv('Advanced_HR_Metrics.csv', index=False)
print("Advanced metrics saved as 'Advanced_HR_Metrics.csv'! Ready for Tableau.")

Advanced metrics saved as 'Advanced_HR_Metrics.csv'! Ready for Tableau.
